# Exploración de las llegadas

Este notebook recorre los conceptos del hito 2 de EDsim ejecutando las funciones del paquete
`edsim`: cómo varía la tasa de llegadas con la hora, el día y el mes, cómo se generan llegadas
con *thinning*, por qué los huecos en minutos no son exponenciales, de dónde sale la
sobredispersión diaria y cómo se valida todo con cinco pruebas estadísticas.

Toda la lógica vive en el paquete y está testeada; aquí solo se llama a sus funciones, se dibuja
y se explica.

## 0. Cómo usar este notebook

- **Kernel:** en Cursor, pulsa *Select Kernel* (arriba a la derecha) y elige el entorno de
  `.venv` del proyecto. Si no aparece, ejecuta antes `uv sync` en la terminal.
- **Ejecutar:** usa *Restart & Run All* (reiniciar y ejecutar todo). El notebook está pensado
  para funcionar de principio a fin sin depender del orden en que ejecutaste las celdas a mano.
- **Semillas:** cada sección fija su semilla en una constante visible. Con la misma semilla
  el resultado es idéntico; cámbiala para ver otra muestra del azar.
- **Duración:** en modo normal tarda unos 3 minutos; casi todo es la validación de la
  sección 8 y las semillas de la sección 9. Para ir más rápido, baja `N_SEMILLAS_C` y `N_META`
  en la celda de parámetros de abajo.
- **Modo rápido:** con la variable de entorno `EDSIM_NB_RAPIDO=1` los días, réplicas y semillas
  se reducen y el notebook tarda segundos. Es lo que hace el CI
  (`EDSIM_NB_RAPIDO=1 uv run pytest --nbmake notebooks/`) para comprobar que se ejecuta sin
  errores; los números de ese modo no sirven para sacar conclusiones.
- **Las gráficas desaparecen tras cada commit:** el hook `nbstripout` de pre-commit borra
  resultados e imágenes antes de guardar el notebook en git. Vuelve a ejecutarlo para verlas.
- **El perfil** se lee de `config/llegadas.toml`, el mismo que usa la CLI. El notebook se
  ejecuta desde la carpeta `notebooks/`, de ahí la ruta `../config/llegadas.toml`.

In [ ]:
import dataclasses
import os
import tomllib
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from matplotlib.patches import Patch
from scipy import stats

from edsim.arrivals.faults import midnight_restart_fault
from edsim.arrivals.profile import DAYS_PER_MONTH, ArrivalProfile, load_profile
from edsim.arrivals.rate import MINUTES_PER_DAY, cumulative_rate, day_scale, rate
from edsim.arrivals.report import (
    OBSERVED_COLOR,
    THEORY_COLOR,
    WEEKDAYS,
    format_validation,
    plot_daily_residuals,
    plot_hourly_profile,
    plot_rescaled_qq,
)
from edsim.arrivals.thinning import (
    daily_count_distribution,
    day_effect_distribution,
    draw_day_effects,
    iter_arrivals,
    simulate_arrivals,
)
from edsim.arrivals.validation import (
    ALPHA,
    check_hourly_shape,
    check_rescaled_gaps,
    check_volume,
    daily_residuals,
    expected_per_day,
    hour_of_day,
    interval_counts,
    p_value_uniformity,
    pearson_index,
    rescaled_gaps,
    validate_arrivals,
)
from edsim.seeds import child_streams

# --- Parámetros ------------------------------------------------------------------------
# Con EDSIM_NB_RAPIDO=1 (el CI) todo se reduce para que el notebook tarde segundos.
RAPIDO = os.environ.get("EDSIM_NB_RAPIDO") == "1"

CONFIG = Path("../config/llegadas.toml")  # el notebook se ejecuta desde notebooks/
DIAS = 28 if RAPIDO else 364  # días de cada ejecución larga (secciones 4 y 6 a 10)
REPLICAS = 3 if RAPIDO else 20  # réplicas de la prueba d (secciones 8 y 10)
CORTAS, DIAS_CORTOS = (10, 7) if RAPIDO else (100, 14)  # ejecuciones de la prueba e
REPLICAS_DISP = 2 if RAPIDO else 10  # sección 7: años para medir la dispersión
BASES_VOLUMEN = [150, 300] if RAPIDO else [100, 200, 300, 400, 600]  # sección 7
REPLICAS_VOLUMEN = 1 if RAPIDO else 3  # sección 7: años por cada base
N_META = 20 if RAPIDO else 300  # sección 9: ejecuciones cortas para la meta-prueba
N_SEMILLAS_C = 3 if RAPIDO else 50  # sección 9: semillas del error C′

PERFIL = load_profile(CONFIG)
SIN_Z = dataclasses.replace(PERFIL, daily_cv=0)  # el mismo perfil sin efecto diario
DIA = MINUTES_PER_DAY  # 1440 minutos

# Colores de report.py (paleta del proyecto): observado en azul, teoría en naranja; una
# tercera serie en aqua, lo descartado en gris y el pasa/falla con sus colores de estado.
AQUA, GRIS = "#1baf7a", "#a3a29c"
PASA, FALLA = "#0ca30c", "#d03b3b"

plt.rcParams["figure.figsize"] = (9, 4)
print("Modo rápido" if RAPIDO else "Modo normal")

## 1. Introducción: de una λ constante a λ(t)

En la M/M/1 del hito 1 los clientes llegaban con una **tasa constante** λ: los tiempos entre
llegadas eran exponenciales de media 1/λ, a cualquier hora. Un centro de 300 urgencias al día
con tasa constante recibiría una cada 1440/300 = 4,8 minutos de media, igual a las 4 de la
madrugada que a las 11 de la mañana.

Las urgencias reales no son así. La tasa es un **grifo que se abre y se cierra**: casi cerrado de
madrugada, abierto del todo a media mañana. El modelo es un **proceso de Poisson no
homogéneo**: las llegadas siguen siendo independientes, pero con una tasa λ(t) que cambia con el
tiempo; en un intervalo corto dt llega alguien con probabilidad λ(t)·dt.

El paquete lo modela como un producto de factores (el reloj va en minutos):

> λ(t) = B/1440 × h(hora) × d(día de la semana) × m(mes) × Z(día)

B es la base diaria (300), h, d y m tienen media 1 y Z es un efecto aleatorio diario (sección 7).
Abajo, un lunes de enero con un perfil **plano** (todos los factores a 1) frente al perfil real
sin Z.

In [ ]:
SEMILLA = 1
plano = ArrivalProfile(
    daily_base=PERFIL.daily_base,
    hour_factors=(1,) * 24,
    weekday_factors=(1,) * 7,
    month_factors=(1,) * 12,
    daily_cv=0,
    start_date=PERFIL.start_date,
)
llegadas_plano, _ = simulate_arrivals(plano, 1, SEMILLA)
llegadas_real, _ = simulate_arrivals(SIN_Z, 1, SEMILLA)

for nombre, llegadas in [("plano", llegadas_plano), ("real", llegadas_real)]:
    horas = hour_of_day([a.time for a in llegadas])
    madrugada, mediodia = np.sum((horas >= 3) & (horas < 5)), np.sum((horas >= 11) & (horas < 13))
    print(
        f"{nombre:>5}: {len(llegadas)} llegadas;"
        f" entre 3 y 5 h: {madrugada}, entre 11 y 13 h: {mediodia}"
    )

In [ ]:
t = np.arange(0, DIA, 1.0)
fig, (ax, ax_llegadas) = plt.subplots(
    2, 1, sharex=True, figsize=(9, 5), gridspec_kw={"height_ratios": [3, 1]}
)
ax.plot(t / 60, rate(t, SIN_Z), color=OBSERVED_COLOR, lw=2, label="perfil real: λ(t)")
ax.plot(t / 60, rate(t, plano), color=THEORY_COLOR, lw=2, label="perfil plano: λ constante")
ax.set_ylabel("llegadas por minuto")
ax.set_title("Lunes 5 de enero: la tasa es un grifo que se abre y se cierra")
ax.grid(alpha=0.3)
ax.legend()
ax_llegadas.eventplot(
    [[a.time / 60 for a in llegadas_plano], [a.time / 60 for a in llegadas_real]],
    colors=[THEORY_COLOR, OBSERVED_COLOR],
    lineoffsets=[0, 1],
    linelengths=0.8,
)
ax_llegadas.set_yticks([0, 1], ["plano", "real"])
ax_llegadas.set_xlabel("hora del día")
ax_llegadas.set_xticks(range(0, 25, 2))
plt.show()

**Para experimentar**

- ¿Por qué el lunes de enero tiene más llegadas que la base de 300? (Pista: d y m, sección 2.)
- Cambia `SEMILLA`. ¿Cambia mucho el número de llegadas entre 3 y 5 h? ¿Y entre 11 y 13 h?

> **Lo que debes recordar:** con λ(t) las llegadas siguen siendo independientes, pero su ritmo cambia con la hora, el día y el mes; λ(t)·dt es la probabilidad de una llegada en un intervalo corto dt.

## 2. Perfiles y calibración

El perfil está en `config/llegadas.toml`, con claves en español y en términos que entiende un
clínico: el **porcentaje** de las llegadas del día en cada hora, el de la semana en cada día y
un **factor de tasa** por mes. `load_profile` los valida y los normaliza a factores de media 1:

- h = 24 × reparto / suma y d = 7 × reparto / suma, así que no hace falta que sumen 100.
- El factor mensual se normaliza para que su media **ponderada por los días** de cada mes sea 1.

Con media 1, la base conserva su significado: 300 urgencias al día de media anual.

La forma se contrastó con el CMBD de urgencias del Servicio Andaluz de Salud de 2012: picos de
10 a 13 h y de 16 a 17 h, valle de 2 a 8 h; lunes 16,27 %, sábado 13,08 % y domingo 13,58 %
de la semana; enero 9,31 % del año y septiembre 7,72 %. Las sombras y los puntos de la gráfica
son esos datos publicados; el resto de valores son supuestos razonables.

In [ ]:
with CONFIG.open("rb") as f:
    crudo = tomllib.load(f)

print(
    f"reparto_horario suma {sum(crudo['reparto_horario']):.2f} %;"
    f" reparto_semanal suma {sum(crudo['reparto_semanal']):.2f} %"
)
h, d, m = (np.array(f) for f in (PERFIL.hour_factors, PERFIL.weekday_factors, PERFIL.month_factors))
print(
    f"media de h = {h.mean():.4f}; máximo {h.max():.2f} a las {h.argmax()} h,"
    f" mínimo {h.min():.3f} a las {h.argmin()} h"
)
print(f"media de d = {d.mean():.4f}; lunes {d[0]:.3f}, sábado {d[5]:.3f}")
print(
    f"media de m ponderada por días = {np.sum(m * DAYS_PER_MONTH) / 365:.4f};"
    f" media simple = {m.mean():.4f}"
)

**La trampa del porcentaje mensual.** El estudio da enero como el 9,31 % de las urgencias del
año. Dividirlo entre 100/12 = 8,33 % daría un factor de 1,117, pero enero tiene 31 días, más que
un mes medio (365/12 = 30,4): parte de su 9,31 % se debe solo a su duración. El factor de
**tasa diaria** es 9,31 % × 365 / 31 = 1,096. Septiembre, con 30 días, pasa de 0,926 a 0,939.

In [ ]:
SAS_SEMANA = {0: 16.27, 5: 13.08, 6: 13.58}  # % de la semana: lunes, sábado y domingo
SAS_MES = {0: 9.31, 8: 7.72}  # % del año: enero y septiembre

# El porcentaje del año pasado a factor de tasa diaria (correcto) y dividido entre 100/12 (trampa).
tasa_diaria = {k: p / 100 * 365 / DAYS_PER_MONTH[k] for k, p in SAS_MES.items()}
trampa = {k: p / (100 / 12) for k, p in SAS_MES.items()}
for k in SAS_MES:
    print(
        f"mes {k + 1}: tasa diaria {tasa_diaria[k]:.3f}, trampa {trampa[k]:.3f}, perfil {m[k]:.3f}"
    )

In [ ]:
fig, (ax_h, ax_d, ax_m) = plt.subplots(
    1, 3, figsize=(13, 4), gridspec_kw={"width_ratios": [3, 1.3, 2]}
)

ax_h.axvspan(1.5, 7.5, color=GRIS, alpha=0.25)
for inicio, fin in [(9.5, 12.5), (15.5, 16.5)]:
    ax_h.axvspan(inicio, fin, color=THEORY_COLOR, alpha=0.25)
ax_h.bar(range(24), crudo["reparto_horario"], color=OBSERVED_COLOR)
ax_h.set_xlabel("hora")
ax_h.set_ylabel("% de las llegadas del día")
ax_h.set_title("Horas")
ax_h.legend(
    handles=[
        Patch(color=OBSERVED_COLOR, label="reparto del perfil"),
        Patch(color=GRIS, alpha=0.25, label="valle SAS (2–8 h)"),
        Patch(color=THEORY_COLOR, alpha=0.25, label="picos SAS (10–13 y 16–17 h)"),
    ],
    fontsize=8,
)

ax_d.bar(range(7), crudo["reparto_semanal"], color=OBSERVED_COLOR)
ax_d.plot(
    list(SAS_SEMANA), list(SAS_SEMANA.values()), "o", color=THEORY_COLOR, ms=8, label="SAS 2012"
)
ax_d.set_xticks(range(7), [w[:3] for w in WEEKDAYS])
ax_d.set_ylim(12, 17)
ax_d.set_ylabel("% de la semana")
ax_d.set_title("Días")
ax_d.legend(fontsize=8)

ax_m.bar(range(12), m, color=OBSERVED_COLOR, label="factor del perfil")
ax_m.plot(
    list(tasa_diaria),
    list(tasa_diaria.values()),
    "o",
    color=THEORY_COLOR,
    ms=8,
    label="SAS como tasa diaria",
)
ax_m.plot(
    list(trampa),
    list(trampa.values()),
    "x",
    color="k",
    ms=8,
    mew=2,
    label="SAS dividido entre 100/12",
)
ax_m.axhline(1, color="k", lw=0.8, ls=":")
ax_m.set_xticks(range(12), list("EFMAMJJASOND"))
ax_m.set_ylim(0.85, 1.15)
ax_m.set_ylabel("factor de tasa diaria")
ax_m.set_title("Meses")
ax_m.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Para experimentar**

- Construye un perfil con `dataclasses.replace(PERFIL, hour_factors=[x * 2 for x in crudo["reparto_horario"]])`.
  ¿Cambian los factores normalizados? ¿Por qué?
- Si febrero tuviera el 7,5 % del año, ¿qué factor de tasa diaria le correspondería?

> **Lo que debes recordar:** los repartos son porcentajes y el código los convierte en factores de media 1 para que la base conserve su significado; un porcentaje mensual se pasa a tasa diaria dividiendo por los días del mes, no por 100/12.

## 3. λ(t) y Λ(t)

**λ(t)** es la tasa instantánea, en llegadas por minuto. **Λ(t)**, la intensidad acumulada, es
el área bajo λ entre 0 y t: el número **esperado** de llegadas hasta t. Las llegadas entre a y b
siguen una Poisson de media Λ(b) − Λ(a); el área de un día completo es B × d × m.

**Escalones frente a interpolación.** El valor de cada hora corresponde al centro de su hora
(h:30) y los centros se unen con rectas. Con escalones, λ saltaría en cada hora en punto sin
motivo; con rectas es continua y el área de cada día es la misma. El día de la semana y el mes
cambian a medianoche: ahí, y solo ahí, λ puede saltar.

In [ ]:
SEMANA = 7 * DIA
t = np.arange(0, SEMANA + 1, 1.0)
lam = rate(t, SIN_Z)
acumulada = cumulative_rate(t, SIN_Z)

bordes_dias = np.arange(8) * DIA
por_dia = np.diff(cumulative_rate(bordes_dias, SIN_Z))
for k in range(7):
    b_d_m = day_scale(k, SIN_Z) * DIA
    print(f"{WEEKDAYS[k]:>9}: Λ del día = {por_dia[k]:.2f}   B·d·m = {b_d_m:.2f}")

In [ ]:
fig, (ax_l, ax_a) = plt.subplots(2, 1, sharex=True, figsize=(10, 6))
ax_l.plot(t / DIA, lam, color=OBSERVED_COLOR, lw=1.5)
ax_l.fill_between(t / DIA, lam, color=OBSERVED_COLOR, alpha=0.2)
ax_l.set_ylabel("λ(t), llegadas por minuto")
ax_l.set_title("Una semana de enero sin Z: el área bajo λ es Λ")
ax_a.plot(t / DIA, acumulada, color=OBSERVED_COLOR, lw=2)
ax_a.plot(
    bordes_dias / DIA,
    cumulative_rate(bordes_dias, SIN_Z),
    "o",
    color=THEORY_COLOR,
    label="Λ al final de cada día",
)
ax_a.set_ylabel("Λ(t), llegadas esperadas")
ax_a.set_xticks(range(8), [w[:3] for w in WEEKDAYS] + ["lun"])
ax_a.set_xlabel("día (medianoche a la izquierda de cada etiqueta)")
ax_a.legend()
for ax in (ax_l, ax_a):
    ax.grid(alpha=0.3)
plt.show()

In [ ]:
# Escalones: cada hora, su factor horario durante toda la hora.
t2 = np.arange(18 * 60, DIA + 6 * 60, 1.0)  # de las 18 h del lunes a las 6 h del martes
dias_t2 = (t2 // DIA).astype(int)
escalones = day_scale(dias_t2, SIN_Z) * h[hour_of_day(t2)]
rectas = rate(t2, SIN_Z)

t_dia = np.arange(0, DIA, 1.0)
area_escalones = np.sum(day_scale(0, SIN_Z) * h[hour_of_day(t_dia)])  # 1 minuto por paso
print(f"Área del lunes: escalones {area_escalones:.3f}, rectas {cumulative_rate(DIA, SIN_Z):.3f}")

fig, ax = plt.subplots()
ax.plot(t2 / 60, escalones, color=THEORY_COLOR, lw=1.5, label="escalones")
ax.plot(t2 / 60, rectas, color=OBSERVED_COLOR, lw=2, label="interpolación lineal (λ del paquete)")
ax.axvline(24, color="k", ls=":", lw=1)
ax.text(24.1, rectas.max(), "medianoche: cambia d", va="top", fontsize=9)
ax.set_xticks(range(18, 31), [f"{x % 24}" for x in range(18, 31)])
ax.set_xlabel("hora")
ax.set_ylabel("llegadas por minuto")
ax.set_title("Lunes a martes: los escalones saltan cada hora; las rectas, solo a medianoche")
ax.legend()
plt.show()

**Para experimentar**

- ¿En qué hora en punto es mayor el salto de los escalones? ¿Coincide con el mayor cambio del reparto horario?
- El martes tiene menos llegadas que el lunes: ¿cuánto vale el salto de λ a medianoche en
  proporción? Compáralo con d(martes)/d(lunes).

> **Lo que debes recordar:** λ(t) es la tasa y Λ(t), su área: el número esperado de llegadas hasta t; con la interpolación lineal λ es continua dentro del día, el área diaria es B·d·m y Λ es exacta con trapecios.

## 4. *Thinning* paso a paso

¿Cómo se generan llegadas con una tasa que cambia? El algoritmo de **Lewis–Shedler** no
necesita integrar ni invertir Λ:

1. Se elige una **cota** λ\* que λ(t) nunca supera; aquí, el máximo de λ en cada día.
2. Se generan **candidatas** como si la tasa fuera constante e igual a la cota: huecos
   exponenciales de media 1/λ\*.
3. Cada candidata se **acepta** con probabilidad λ(t)/λ\*: se sortea U uniforme y se acepta si
   U·λ\* ≤ λ(t), es decir, si el punto (t, U·λ\*) cae bajo la curva.

El resultado es **exacto**: las aceptadas son un Poisson de tasa λ(t). El gancho `on_candidate`
del generador entrega cada candidata con su altura U·λ\*, sin cambiar las llegadas.

In [ ]:
SEMILLA = 4
candidatas, cierre = [], []
llegadas_dia = list(
    iter_arrivals(PERFIL, 1, SEMILLA, on_day=cierre.append, on_candidate=candidatas.append)
)
lunes = cierre[0]
tiempos = np.array([c.time for c in candidatas])
alturas = np.array([c.height for c in candidatas])
aceptada = np.array([c.accepted for c in candidatas])
print(f"Z = {lunes.day_effect:.3f}; cota = {lunes.bound:.4f} llegadas por minuto")
print(
    f"{lunes.candidates} candidatas, {lunes.accepted} aceptadas"
    f" ({lunes.accepted / lunes.candidates:.1%})"
)

t = np.arange(0, DIA, 1.0)
lam_lunes = rate(t, PERFIL, day_effects=[lunes.day_effect])
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.scatter(tiempos[~aceptada] / 60, alturas[~aceptada], s=8, color=GRIS, label="descartada")
ax.scatter(tiempos[aceptada] / 60, alturas[aceptada], s=8, color=OBSERVED_COLOR, label="aceptada")
ax.plot(t / 60, lam_lunes, color=THEORY_COLOR, lw=2, label="λ(t)")
ax.axhline(lunes.bound, color="k", ls="--", lw=1, label="cota λ*")
ax.set_xticks(range(0, 25, 2))
ax.set_xlabel("hora del día")
ax.set_ylabel("altura U·λ* (llegadas por minuto)")
ax.set_title("Thinning de un lunes: se aceptan las candidatas que caen bajo λ(t)")
ax.legend(loc="upper left", fontsize=8)
plt.show()

**La eficiencia.** La proporción aceptada es el área bajo λ dividida entre el área bajo la cota:
media(h)/máx(h) = 1/1,68 = 59,5 %, igual todos los días, porque d, m y Z multiplican a la vez a
λ y a la cota. Una cota más alta seguiría siendo exacta, pero desperdiciaría más candidatas.

In [ ]:
_, dias_ano = simulate_arrivals(PERFIL, DIAS, SEMILLA)
eficiencia = np.array([dd.accepted / dd.candidates for dd in dias_ano])
total = sum(dd.accepted for dd in dias_ano) / sum(dd.candidates for dd in dias_ano)
print(f"{DIAS} días: aceptadas / candidatas = {total:.4f}; teoría 1/máx(h) = {1 / h.max():.4f}")

fig, ax = plt.subplots()
ax.hist(eficiencia, bins=30, color=OBSERVED_COLOR, edgecolor="white")
ax.axvline(1 / h.max(), color=THEORY_COLOR, lw=2, label="1 / máx(h)")
ax.set_xlabel("proporción de candidatas aceptadas en el día")
ax.set_ylabel("días")
ax.set_title("Eficiencia del thinning, día a día")
ax.legend()
plt.show()

**Una cota insuficiente.** Si la cota quedara por debajo de λ en algún momento, el algoritmo no
fallaría: aceptaría todas las candidatas de esa zona y **recortaría el pico** sin avisar. Por
eso el generador comprueba la cota en cada candidata y lanza un error. Con `bound_factor=0.8`
(una cota un 20 % baja), el error salta en cuanto una candidata cae donde λ supera la cota.

In [ ]:
FACTOR = 0.8
try:
    list(iter_arrivals(PERFIL, 1, SEMILLA, bound_factor=FACTOR))
except RuntimeError as error:
    print(f"RuntimeError: {error}")

cota_baja = FACTOR * lunes.bound
recorte = np.maximum(lam_lunes - cota_baja, 0)  # 1 minuto por paso: suma = área
print(
    f"Sin la comprobación se perderían {recorte.sum():.1f} de {lam_lunes.sum():.1f} llegadas"
    f" esperadas ({recorte.sum() / lam_lunes.sum():.1%}), todas en el pico."
)

fig, ax = plt.subplots()
ax.plot(t / 60, lam_lunes, color=THEORY_COLOR, lw=2, label="λ(t)")
ax.axhline(cota_baja, color="k", ls="--", lw=1, label=f"cota × {FACTOR}")
ax.fill_between(
    t / 60,
    cota_baja,
    lam_lunes,
    where=lam_lunes > cota_baja,
    color=FALLA,
    alpha=0.3,
    label="llegadas que se recortarían",
)
ax.set_xticks(range(0, 25, 2))
ax.set_xlabel("hora del día")
ax.set_ylabel("llegadas por minuto")
ax.legend()
plt.show()

**Para experimentar**

- Con `bound_factor=2` en `iter_arrivals`, ¿qué eficiencia esperas? ¿Cambia el número medio de llegadas?
- Prueba `FACTOR = 0.99`. ¿Salta siempre el error? ¿Por qué podría no saltar con otra semilla?

> **Lo que debes recordar:** el thinning genera candidatas a la tasa constante de la cota y acepta cada una con probabilidad λ(t)/cota; es exacto con cualquier cota que domine a λ, su eficiencia es media(h)/máx(h) ≈ 60 %, y una cota insuficiente recortaría los picos, por eso se comprueba.

## 5. Una semana simulada

Una semana simulada, contada hora a hora (`interval_counts`), frente a lo esperado: Λ(fin) −
Λ(inicio) de cada hora, con la Z de cada día. Cada recuento es una Poisson con esa media, así
que su desviación típica es √esperado: unas ±4 llegadas en una hora de 20, ±1,5 en una de 2.

In [ ]:
SEMILLA = 5
llegadas_sem, dias_sem = simulate_arrivals(PERFIL, 7, SEMILLA)
bordes, observadas, esperadas = interval_counts(PERFIL, llegadas_sem, dias_sem)
residuos = (observadas - esperadas) / np.sqrt(esperadas)
peor = np.argmax(np.abs(observadas - esperadas))
print(f"{observadas.sum()} llegadas frente a {esperadas.sum():.1f} esperadas")
print(
    f"Mayor desviación: {observadas[peor]} frente a {esperadas[peor]:.1f}"
    f" ({WEEKDAYS[peor // 24]} a las {peor % 24} h)"
)
print(f"Residuos (o − e)/√e: media {residuos.mean():.2f}, desviación típica {residuos.std():.2f}")

fig, ax = plt.subplots(figsize=(11, 4))
ax.stairs(observadas, bordes / DIA, fill=True, color=OBSERVED_COLOR, alpha=0.6, label="observadas")
ax.plot((bordes[:-1] + 30) / DIA, esperadas, color=THEORY_COLOR, lw=2, label="esperadas")
ax.set_xticks(range(8), [w[:3] for w in WEEKDAYS] + ["lun"])
ax.set_ylabel("llegadas en la hora")
ax.set_title(f"Una semana hora a hora (semilla {SEMILLA})")
ax.legend()
plt.show()

In [ ]:
esperadas_sin_z = np.diff(cumulative_rate(bordes, PERFIL))  # Z tiene media 1
fig, ax = plt.subplots(figsize=(11, 4))
for semilla in range(SEMILLA, SEMILLA + 5):
    ll, dd = simulate_arrivals(PERFIL, 7, semilla)
    ax.stairs(interval_counts(PERFIL, ll, dd)[1], bordes / DIA, color=OBSERVED_COLOR, alpha=0.35)
ax.plot([], [], color=OBSERVED_COLOR, alpha=0.35, label="cinco semillas")
ax.plot(
    (bordes[:-1] + 30) / DIA, esperadas_sin_z, color=THEORY_COLOR, lw=2, label="esperadas sin Z"
)
ax.set_xticks(range(8), [w[:3] for w in WEEKDAYS] + ["lun"])
ax.set_ylabel("llegadas en la hora")
ax.set_title("El mismo modelo, cinco semanas distintas")
ax.legend()
plt.show()

**Para experimentar**

- ¿En qué horas son mayores las desviaciones absolutas? ¿Y las relativas? ¿Por qué no coinciden?
- Cambia `interval_counts(..., width=30)`. ¿Qué pasa con el ruido relativo de cada barra?

> **Lo que debes recordar:** cada recuento horario es una Poisson de media Λ(fin) − Λ(inicio): la desviación típica es √esperado, mayor en términos absolutos en las horas punta y mayor en términos relativos de madrugada.

## 6. Huecos y reescalado

En la M/M/1 los tiempos entre llegadas eran exponenciales. Con λ(t) variable **ya no lo son**:
la muestra mezcla huecos largos de madrugada (media ≈ 1/λ pequeña) y cortos a mediodía. Una
mezcla de exponenciales de medias distintas tiene más huecos muy cortos y muy largos que una
sola exponencial: su coeficiente de variación pasa de 1.

El **teorema del reescalado temporal** lo arregla: si las llegadas son un Poisson de tasa λ(t),
los tiempos Λ(tᵢ) son un Poisson de tasa 1, y sus huecos Λ(tᵢ) − Λ(tᵢ₋₁) son exponenciales de
media 1. Es la base de la prueba c de la validación.

In [ ]:
SEMILLA = 6
llegadas_6, dias_6 = simulate_arrivals(PERFIL, DIAS, SEMILLA)
tiempos_6 = np.array([a.time for a in llegadas_6])
huecos = np.diff(tiempos_6)
reescalados = rescaled_gaps(PERFIL, llegadas_6, dias_6)
horas_6 = hour_of_day(tiempos_6[1:])

print(
    f"Huecos en minutos: media {huecos.mean():.2f}, CV {huecos.std() / huecos.mean():.3f}"
    " (exponencial: CV = 1)"
)
print(
    f"  de 3 a 5 h: media {huecos[(horas_6 >= 3) & (horas_6 < 5)].mean():.2f} min;"
    f" de 11 a 13 h: {huecos[(horas_6 >= 11) & (horas_6 < 13)].mean():.2f} min"
)
print(
    f"Huecos reescalados: media {reescalados.mean():.3f},"
    f" CV {reescalados.std() / reescalados.mean():.3f}"
)

In [ ]:
n = huecos.size
cuantiles = stats.expon.ppf((np.arange(1, n + 1) - 0.5) / n, scale=huecos.mean())
tope = max(huecos.max(), cuantiles.max())
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.plot(
    cuantiles, np.sort(huecos), ".", color=OBSERVED_COLOR, ms=3, label=f"huecos en minutos ({n})"
)
ax.plot(
    [0, tope],
    [0, tope],
    color=THEORY_COLOR,
    lw=2,
    label=f"diagonal: Exp de media {huecos.mean():.2f}",
)
ax.set_xlim(0, tope)
ax.set_ylim(0, tope)
ax.set_aspect("equal")
ax.set_xlabel("cuantil teórico de la exponencial")
ax.set_ylabel("hueco en minutos")
ax.set_title("Huecos en minutos frente a una exponencial")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

In [ ]:
display(plot_rescaled_qq(PERFIL, llegadas_6, dias_6))

**Para experimentar**

- Repite con el perfil `plano` de la sección 1 (y `daily_cv=0`): ¿son ahora exponenciales los
  huecos en minutos? ¿Cuánto vale su CV?
- ¿Qué parte de la curva QQ en minutos se aleja más de la diagonal, la cola de los huecos
  largos o la de los cortos? ¿A qué horas pertenecen esos huecos?

> **Lo que debes recordar:** con λ(t) variable, los huecos en minutos mezclan exponenciales de medias distintas y no son exponenciales; reescalados con Λ(tᵢ) vuelven a ser Exp(1), y eso es lo que contrasta la prueba c.

## 7. Sobredispersión

Si cada día fuera un Poisson puro, la varianza del recuento diario sería igual a su media. Los
datos reales varían más: hay días malos y buenos por causas que el modelo no ve (el tiempo, una
epidemia, un partido). EDsim lo modela con un **efecto diario Z**, gamma de media 1 y
coeficiente de variación CV = 0,06, que multiplica λ durante todo el día.

- **Dado Z**, el día es un Poisson de media Z·e (e: llegadas esperadas del día sin Z).
- **Sin conocer Z**, el recuento es una **binomial negativa** de media e y varianza e + (CV·e)².

Z es *ground truth*: queda en `DayRecord.day_effect` y el hospital nunca lo observa.

In [ ]:
SEMILLA = 7
z = draw_day_effects(PERFIL.daily_cv, DIAS, SEMILLA)
gamma = day_effect_distribution(PERFIL.daily_cv)
print(
    f"{DIAS} valores de Z: media {z.mean():.4f}, CV {z.std() / z.mean():.4f}"
    f" (teoría: 1 y {PERFIL.daily_cv})"
)

fig, (ax_t, ax_h) = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [2, 1]})
ax_t.plot(np.arange(DIAS), z, color=OBSERVED_COLOR, lw=1)
ax_t.axhline(1, color="k", ls=":", lw=1)
ax_t.set_xlabel("día")
ax_t.set_ylabel("Z")
ax_t.set_title("Efecto diario: independiente de un día a otro")
rejilla_z = np.linspace(z.min(), z.max(), 200)
ax_h.hist(z, bins=25, density=True, color=OBSERVED_COLOR, edgecolor="white", label="sorteadas")
ax_h.plot(rejilla_z, gamma.pdf(rejilla_z), color=THEORY_COLOR, lw=2, label="gamma")
ax_h.set_xlabel("Z")
ax_h.legend()
plt.show()

**¿Por qué no comparar los recuentos crudos con una binomial negativa?** Porque no hay *una*:
cada día tiene su propia e (los lunes de enero esperan unas 375 llegadas; los sábados de
septiembre, unas 245). El histograma de los recuentos de un año mezcla días de medias distintas
y es mucho más ancho que la binomial negativa de la media ē. Lo que sí encaja es la **mezcla**:
la media de las binomiales negativas de todos los días.

In [ ]:
replicas_7 = [
    simulate_arrivals(PERFIL, DIAS, hijo)[1] for hijo in child_streams(SEMILLA, REPLICAS_DISP)
]
e = expected_per_day(PERFIL, DIAS)
recuentos = np.concatenate([[dd.accepted for dd in r] for r in replicas_7])
una_sola = daily_count_distribution(e.mean(), PERFIL.daily_cv)
k = np.arange(recuentos.min() - 20, recuentos.max() + 21)
mezcla = daily_count_distribution(e, PERFIL.daily_cv).pmf(k[:, None]).mean(axis=1)

print(f"Recuentos crudos: media {recuentos.mean():.1f}, varianza {recuentos.var():.0f}")
print(f"Binomial negativa de ē = {e.mean():.1f}: varianza {una_sola.var():.0f}")
print(f"Mezcla de los {DIAS} días: varianza {np.sum(mezcla * (k - np.sum(mezcla * k)) ** 2):.0f}")

fig, ax = plt.subplots()
ax.hist(
    recuentos,
    bins=40,
    density=True,
    color=OBSERVED_COLOR,
    edgecolor="white",
    label=f"recuentos crudos ({recuentos.size} días)",
)
ax.plot(k, una_sola.pmf(k), color=THEORY_COLOR, lw=2, label="una sola binomial negativa (ē)")
ax.plot(k, mezcla, color=AQUA, lw=2, label="mezcla: una binomial negativa por día")
ax.set_xlabel("llegadas del día")
ax.set_ylabel("densidad")
ax.legend()
plt.show()

**La solución: estandarizar cada día con su propia media.** Los residuos estandarizados
(gráfica de `edsim llegadas validar`) tienen media 0 y varianza 1 bajo la hipótesis:

- **Con Z** (condicional): (c − e·Z)/√(e·Z), el Poisson de cada día.
- **Sin Z**: (c − e)/√(e + CV²·e²), la binomial negativa.

**Pearson frente al residuo.** El índice de Pearson sin Z, P = media de (c − e)²/e, divide por
la varianza de **Poisson**, e, así que mide cuánto se aleja el día de un Poisson puro: vale
1 + CV²·ē ≈ 2,08. El residuo sin Z divide por la varianza **verdadera**, e + CV²·e², y su
varianza vale ≈ 1 si el CV es el correcto. Son la misma información desde dos puntos de vista:
P dice "hay sobredispersión"; el residuo, "el modelo la explica".

In [ ]:
display(plot_daily_residuals(PERFIL, replicas_7))

partes = [daily_residuals(PERFIL, r) for r in replicas_7]
condicional = np.concatenate([c for c, _ in partes])
residuo_sin_z = np.concatenate([u for _, u in partes])
pearson = np.mean([pearson_index(PERFIL, r) for r in replicas_7])
print(
    f"Pearson sin Z = {pearson:.3f}  (teoría 1 + CV²·ē = {1 + PERFIL.daily_cv**2 * e.mean():.3f})"
)
print(
    f"Varianza del residuo sin Z = {residuo_sin_z.var():.3f};"
    f" del residuo condicional = {condicional.var():.3f}"
)

**El índice de dispersión crece con el volumen.** Como P = 1 + CV²·ē, un mismo CV da un índice
mayor en un centro grande. Por eso EDsim calibra el **CV**, que se puede trasladar entre centros,
y no el índice: Whitt y Zhang midieron índices de 1,2 a 1,5 con unas 134 urgencias al día, y con
CV 0,06 un centro de 134 da 1 + 0,0036 × 134 = 1,48.

In [ ]:
medidos = {}
for base in BASES_VOLUMEN:
    perfil_b = dataclasses.replace(PERFIL, daily_base=base)
    medidos[base] = [
        pearson_index(perfil_b, simulate_arrivals(perfil_b, DIAS, hijo)[1])
        for hijo in child_streams(SEMILLA, REPLICAS_VOLUMEN)
    ]

bases = np.linspace(50, max(BASES_VOLUMEN) + 50, 100)
fig, ax = plt.subplots()
ax.plot(
    bases,
    1 + PERFIL.daily_cv**2 * bases,
    color=THEORY_COLOR,
    lw=2,
    label=f"teoría con CV {PERFIL.daily_cv}: 1 + CV²·base",
)
ax.axhline(1, color="k", ls=":", lw=1, label="Poisson puro (CV 0)")
for base, valores in medidos.items():
    ax.plot([base] * len(valores), valores, "o", color=OBSERVED_COLOR, ms=8)
ax.plot([], [], "o", color=OBSERVED_COLOR, label=f"medido ({REPLICAS_VOLUMEN} por base)")
ax.set_xlabel("base diaria")
ax.set_ylabel("índice de dispersión de Pearson")
ax.set_title("Mismo CV, índice creciente con el volumen")
ax.legend()
plt.show()

**Para experimentar**

- ¿Qué índice daría un centro de 134 urgencias con CV 0,06? ¿Y qué CV haría falta para un índice
  de 1,3 en ese centro?
- Con `PERFIL.daily_cv` = 0,12, ¿cuánto valdría P en el centro de 300? Compruébalo con
  `dataclasses.replace(PERFIL, daily_cv=0.12)`.

> **Lo que debes recordar:** el efecto diario Z hace que los recuentos sean binomiales negativas de varianza e + (CV·e)²; como cada día tiene su media, se comparan residuos estandarizados y no recuentos crudos; el Pearson sin Z (≈ 2,08) mide la distancia al Poisson y crece con el volumen, por eso se calibra el CV.

## 8. Validación

Cinco pruebas comprueban que las llegadas son un Poisson no homogéneo con la λ(t) del perfil.
Cada una busca un tipo de error distinto, todas al nivel 0,01:

| Prueba | Qué detecta |
| --- | --- |
| a. Volumen | Escala y normalización: observadas/esperadas, en total y por día de la semana |
| b. Forma horaria | Perfil horario o interpolación: chi-cuadrado por hora del reloj |
| c. Huecos reescalados | Huecos, cota, aceptación, bordes: Kolmogórov–Smirnov frente a Exp(1) |
| d. Dispersión | Z ausente o mal escalado: Pearson sin Z y condicional en réplicas |
| e. Uniformidad de p-valores | Sesgos pequeños: los p-valores de c en muchas ejecuciones cortas |

Es la misma validación que `edsim llegadas validar`, con la misma tabla y las mismas gráficas.

In [ ]:
SEMILLA = 2026
validacion = validate_arrivals(
    PERFIL, DIAS, REPLICAS, SEMILLA, short_runs=CORTAS, short_days=DIAS_CORTOS
)
print(format_validation(validacion.results))

In [ ]:
display(plot_hourly_profile(PERFIL, validacion.arrivals, validacion.days))
display(plot_rescaled_qq(PERFIL, validacion.arrivals, validacion.days))
display(plot_daily_residuals(PERFIL, validacion.replications))

**Potencia: ¿detectan las pruebas un generador equivocado?** Una prueba que nunca falla no
prueba nada. Con `hypothesis=` se generan llegadas con un perfil y se validan contra otro:

- **Base 330 frente a 300:** un 10 % de llegadas de más.
- **Perfil desplazado una hora:** cada hora toma el factor de la anterior; el pico pasa a las 12 h.
- **CV 0,06 validado como 0:** la sobredispersión existe, pero la hipótesis no la incluye.

In [ ]:
h_tupla = PERFIL.hour_factors
CASOS = {
    "base 330 frente a 300": (dataclasses.replace(PERFIL, daily_base=330), PERFIL),
    "perfil desplazado 1 h": (
        dataclasses.replace(PERFIL, hour_factors=h_tupla[-1:] + h_tupla[:-1]),
        PERFIL,
    ),
    "CV 0,06 validado como 0": (PERFIL, SIN_Z),
}
potencia = {
    nombre: validate_arrivals(
        generador,
        DIAS,
        REPLICAS,
        SEMILLA,
        short_runs=CORTAS,
        short_days=DIAS_CORTOS,
        hypothesis=hipotesis,
    )
    for nombre, (generador, hipotesis) in CASOS.items()
}

In [ ]:
COLUMNAS = ["a. volumen", "b. forma", "c. huecos", "d. sin Z", "d. condicional", "e. p-valores"]


def fila(v):
    """Pasa (True) o falla de cada columna, con el estadístico que se muestra en la celda."""
    a, b, c, d, e = v.results
    sin_z_d, cond_d = d.details
    return [
        (a.passed, f"{a.statistic:.3f}"),
        (b.passed, f"χ² {b.statistic:.0f}"),
        (c.passed, f"p {c.p_value:.2g}"),
        (sin_z_d.passed, f"{sin_z_d.statistic:.2f} / {sin_z_d.target:.2f}"),
        (cond_d.passed, f"{cond_d.statistic:.2f} / 1"),
        (e.passed, f"p {e.p_value:.2g}"),
    ]


filas = {"correcto": fila(validacion)} | {nombre: fila(v) for nombre, v in potencia.items()}
fig, ax = plt.subplots(figsize=(11, 3.2))
for i, celdas in enumerate(filas.values()):
    for j, (paso, texto) in enumerate(celdas):
        ax.add_patch(plt.Rectangle((j, i), 0.96, 0.92, color=PASA if paso else FALLA, alpha=0.25))
        ax.text(
            j + 0.48,
            i + 0.46,
            f"{'pasa' if paso else 'FALLA'}\n{texto}",
            ha="center",
            va="center",
            fontsize=8,
            fontweight="normal" if paso else "bold",
        )
ax.set_xlim(0, len(COLUMNAS))
ax.set_ylim(len(filas), 0)
ax.set_xticks(np.arange(len(COLUMNAS)) + 0.48, COLUMNAS)
ax.set_yticks(np.arange(len(filas)) + 0.46, list(filas))
ax.tick_params(length=0)
for lado in ax.spines.values():
    lado.set_visible(False)
ax.set_title("Qué detecta cada prueba (d: observado / teoría)")
plt.show()

Cada prueba ve un tipo de error y ninguna los ve todos:

- **b no ve el volumen**, porque reescala lo esperado al total observado: compara la forma.
- **c ve volumen y forma**, porque reescala con la Λ de la hipótesis, pero no la sobredispersión:
  dado Z, cada día sigue siendo un Poisson.
- **El Pearson condicional pasa con el CV mal especificado**: usa la Z real de los registros y
  el Poisson de cada día está bien. Solo el Pearson sin Z ve que falta la sobredispersión.
- **e hereda lo que ve c**: con la base o el perfil equivocados, los p-valores de c se amontonan
  cerca de 0 también en las ejecuciones cortas, y e falla con ellos.
- Con base 330, el Pearson sin Z sale mucho mayor que 4: el sesgo de un 10 % suma (0,1·e)²/e ≈ 3
  a la varianza. Por eso el volumen tiene su propia prueba.

**Para experimentar**

- ¿Qué pruebas detectan una base de 310 frente a 300? ¿Y de 303? (Cambia `CASOS` y vuelve a
  ejecutar las dos celdas.)
- ¿Detecta b un perfil desplazado solo media hora? Pista: no se puede desplazar media hora
  cambiando el orden de los factores; prueba a promediar cada hora con la anterior.

> **Lo que debes recordar:** cinco pruebas al nivel 0,01, cada una ciega a algo: b no ve el volumen, c no ve la sobredispersión y el Pearson condicional no ve un CV mal puesto; solo el conjunto cubre los errores importantes, y la potencia se comprueba generando con un perfil y validando contra otro.

## 9. La meta-prueba

Con un generador correcto, el **p-valor de una prueba es uniforme** entre 0 y 1: un 5 % de las
semillas dan p < 0,05 y un 1 % dan p < 0,01 **por azar**. Un fallo aislado no prueba nada; lo
que delata un error es que los p-valores se amontonen cerca de 0. La prueba e lo contrasta con
Kolmogórov–Smirnov frente a una uniforme. Abajo, los p-valores de la prueba c en muchas
ejecuciones cortas.

In [ ]:
SEMILLA = 9
p_c = np.array(
    [
        check_rescaled_gaps(PERFIL, *simulate_arrivals(PERFIL, DIAS_CORTOS, hijo)).p_value
        for hijo in child_streams(SEMILLA, N_META)
    ]
)
uniformidad = p_value_uniformity(p_c)
print(
    f"{N_META} ejecuciones de {DIAS_CORTOS} días: p < 0,05 en {np.mean(p_c < 0.05):.1%},"
    f" p < 0,01 en {np.mean(p_c < ALPHA):.1%}"
)
print(
    f"Uniformidad (KS frente a U(0, 1)): D = {uniformidad.statistic:.3f},"
    f" p = {uniformidad.p_value:.3f} → {'pasa' if uniformidad.passed else 'FALLA'}"
)

fig, ax = plt.subplots()
ax.hist(p_c, bins=20, range=(0, 1), color=OBSERVED_COLOR, edgecolor="white", label="p-valores de c")
ax.axhline(N_META / 20, color=THEORY_COLOR, lw=2, label="uniforme: misma altura")
ax.set_xlabel("p-valor")
ax.set_ylabel("ejecuciones")
ax.set_title("Con un generador correcto, los p-valores son uniformes")
ax.legend()
plt.show()

**¿Qué significa una potencia del 13 %?** Al escribir el generador apareció un error, **C′**:
tras la medianoche, el día siguiente arrancaba en el instante de la candidata descartada en vez
de a las 0:00. Se pierden las llegadas de un tramo de unos 2,3 minutos tras cada medianoche,
unas 0,44 al día de media: invisible en el total del año.

`midnight_restart_fault` reproduce el error sobre un run correcto, borrando las llegadas de
[0:00, 0:00 + E) con E exponencial de media 1/cota del día anterior (por la falta de memoria de
la exponencial, es equivalente en distribución). Usa su propio flujo aleatorio, así que el run
correcto y el erróneo comparten todo lo demás.

La tarea 2.4 estimó que la prueba b detecta C′ en un año con una potencia de alrededor del 13 %:
**a veces sí y a veces no**, según la semilla. Lo comprobamos con muchas semillas.

In [ ]:
PRUEBAS = {"a": check_volume, "b": check_hourly_shape, "c": check_rescaled_gaps}
p_valores_b = {"correcto": [], "C′": []}
detecciones = {version: dict.fromkeys(PRUEBAS, 0) for version in p_valores_b}
borradas_por_minuto = np.zeros(15)

for hijo in child_streams(SEMILLA, N_SEMILLAS_C):
    generador, falla = child_streams(hijo, 2)
    llegadas, dias = simulate_arrivals(PERFIL, DIAS, generador)
    llegadas_c, dias_c = midnight_restart_fault(llegadas, dias, falla)
    for version, (ll, dd) in {"correcto": (llegadas, dias), "C′": (llegadas_c, dias_c)}.items():
        resultados = {
            "a": check_volume(PERFIL, dd),
            "b": check_hourly_shape(PERFIL, ll, dd),
            "c": check_rescaled_gaps(PERFIL, ll, dd),
        }
        for prueba, r in resultados.items():
            detecciones[version][prueba] += not r.passed
        p_valores_b[version].append(resultados["b"].p_value)
    # Llegadas borradas en cada minuto tras la medianoche: correcto menos C′.
    por_minuto = (
        interval_counts(PERFIL, llegadas, dias, width=1)[1]
        - interval_counts(PERFIL, llegadas_c, dias_c, width=1)[1]
    )
    borradas_por_minuto += por_minuto.reshape(DIAS, DIA)[:, :15].sum(axis=0)

for version, d_v in detecciones.items():
    print(f"{version:>8}: " + ", ".join(f"{p} falla en {n}/{N_SEMILLAS_C}" for p, n in d_v.items()))
deficit = borradas_por_minuto.sum() / (N_SEMILLAS_C * (DIAS - 1))
print(f"Déficit medio de C′: {deficit:.3f} llegadas al día")
for version, p in p_valores_b.items():
    u = p_value_uniformity(p)
    print(
        f"Uniformidad de los p-valores de b ({version}): p = {u.p_value:.3g}"
        f" → {'pasa' if u.passed else 'FALLA'}"
    )

In [ ]:
fig, (ax_p, ax_m) = plt.subplots(1, 2, figsize=(12, 4))
ax_p.hist(
    [p_valores_b["correcto"], p_valores_b["C′"]],
    bins=20,
    range=(0, 1),
    color=[OBSERVED_COLOR, AQUA],
    label=["correcto", "C′"],
)
ax_p.axvline(ALPHA, color="k", ls=":", lw=1, label="0,01")
ax_p.set_xlabel("p-valor de la prueba b")
ax_p.set_ylabel("semillas")
ax_p.set_title(f"Prueba b en {N_SEMILLAS_C} semillas de {DIAS} días")
ax_p.legend()
ax_m.bar(
    np.arange(15) + 0.5, borradas_por_minuto / (N_SEMILLAS_C * (DIAS - 1)), width=0.9, color=AQUA
)
ax_m.set_xlabel("minuto tras la medianoche")
ax_m.set_ylabel("llegadas borradas por día")
ax_m.set_title("Dónde pierde llegadas C′")
plt.tight_layout()
plt.show()

Una sola semilla responde "sí" o "no"; muchas muestran la **potencia**: la proporción de
semillas en que la prueba detecta el error. Con una potencia baja, que una validación pase no
garantiza que el generador esté bien. Hay dos remedios: la **meta-prueba**, que junta los
p-valores de muchas semillas y ve el amontonamiento cerca de 0 aunque cada uno por separado no
baste, y los **tests de regresión** (la huella SHA-256 de la tarea 2.2), que ven cualquier cambio
en un sitio concreto. Los tests estadísticos ven errores grandes en cualquier parte; los de
regresión, cualquier cambio en un sitio concreto. Hacen falta los dos.

**Para experimentar**

- ¿Cuántas semillas harían falta para estimar la potencia de b con un error de ±5 puntos?
  (Pista: error típico de una proporción, √(p(1 − p)/n).)
- Cambia `DIAS` a 728 en la celda de parámetros. ¿Sube la potencia de b? ¿Por qué?

> **Lo que debes recordar:** con un generador correcto, los p-valores son uniformes y un 1 % de las semillas falla por azar; la potencia es la proporción de semillas que detectan un error, y un error pequeño y localizado como C′ solo se detecta a veces: para él hacen falta la meta-prueba o un test de regresión.

## 10. Para experimentar

Cambia los parámetros y ejecuta las dos celdas. El generador usa `BASE_EXP`, `CV_EXP` y el
perfil horario desplazado `DESPLAZAMIENTO_EXP` horas; la validación compara con la base
`BASE_EXP`, el CV `CV_HIPOTESIS` y el perfil sin desplazar. La tabla es la misma que imprime
`edsim llegadas validar`.

Preguntas guía:

- **Volumen** (`BASE_EXP = 200` o `400`): ¿cambia alguna prueba? ¿Y el Pearson sin Z?
- **Más sobredispersión** (`CV_EXP = CV_HIPOTESIS = 0.12`): ¿qué teoría de d sale?
- **CV mal especificado** (`CV_EXP = 0.06`, `CV_HIPOTESIS = 0`): ¿qué Pearson falla?
- **Pocos días** (`DIAS_EXP = 28`): ¿qué pruebas pierden potencia? ¿Se ensanchan los intervalos?
- **Perfil desplazado** (`DESPLAZAMIENTO_EXP = 1`): ¿qué pruebas fallan? ¿Y con `DIAS_EXP = 28`?

In [ ]:
BASE_EXP = 300
CV_EXP = 0.06  # CV con el que se generan las llegadas
CV_HIPOTESIS = 0.06  # CV con el que se validan
DESPLAZAMIENTO_EXP = 0  # horas que se desplaza el perfil horario del generador
DIAS_EXP, REPLICAS_EXP = DIAS, REPLICAS
SEMILLA_EXP = 2026

k_exp = DESPLAZAMIENTO_EXP % 24
generador = dataclasses.replace(
    PERFIL,
    daily_base=BASE_EXP,
    daily_cv=CV_EXP,
    hour_factors=h_tupla[-k_exp:] + h_tupla[:-k_exp] if k_exp else h_tupla,
)
hipotesis = dataclasses.replace(PERFIL, daily_base=BASE_EXP, daily_cv=CV_HIPOTESIS)
exp = validate_arrivals(
    generador,
    DIAS_EXP,
    REPLICAS_EXP,
    SEMILLA_EXP,
    short_runs=CORTAS,
    short_days=DIAS_CORTOS,
    hypothesis=hipotesis,
)
print(format_validation(exp.results))

In [ ]:
display(plot_hourly_profile(hipotesis, exp.arrivals, exp.days))

> **Lo que debes recordar:** para fiarte de un generador hacen falta varias pruebas que miren
> errores distintos, días suficientes para que tengan potencia y la comprobación de que fallan
> cuando el generador está mal.

## 11. Resumen

Lo que debes recordar de cada sección:

1. Con λ(t) las llegadas siguen siendo independientes, pero su ritmo cambia con la hora, el día
   y el mes; λ(t)·dt es la probabilidad de una llegada en un intervalo corto dt.
2. Los repartos son porcentajes y el código los convierte en factores de media 1 para que la
   base conserve su significado; un porcentaje mensual se pasa a tasa diaria dividiendo por los
   días del mes, no por 100/12.
3. λ(t) es la tasa y Λ(t), su área: el número esperado de llegadas hasta t; con la interpolación
   lineal λ es continua dentro del día y Λ es exacta con trapecios.
4. El thinning genera candidatas a la tasa de la cota y acepta cada una con probabilidad
   λ(t)/cota; es exacto, su eficiencia es media(h)/máx(h) ≈ 60 % y una cota insuficiente
   recortaría los picos, por eso se comprueba.
5. Cada recuento horario es una Poisson de media Λ(fin) − Λ(inicio): desviación típica
   √esperado.
6. Los huecos en minutos mezclan exponenciales de medias distintas y no son exponenciales;
   reescalados con Λ(tᵢ) vuelven a ser Exp(1).
7. Z hace que los recuentos sean binomiales negativas de varianza e + (CV·e)²; se comparan
   residuos estandarizados y no recuentos crudos, y se calibra el CV porque el índice de
   dispersión crece con el volumen.
8. Cinco pruebas, cada una ciega a algo; solo el conjunto cubre los errores importantes, y la
   potencia se comprueba validando contra una hipótesis equivocada.
9. Con un generador correcto los p-valores son uniformes; un error pequeño y localizado solo se
   detecta a veces, y para él hacen falta la meta-prueba o un test de regresión.
10. Para fiarte de un generador hacen falta pruebas que miren errores distintos, días
    suficientes y la comprobación de que fallan cuando debe.

Las tres frases que pide el hito:

- **El thinning es exacto** porque cada candidata de un Poisson de tasa constante λ\* se conserva
  con probabilidad λ(t)/λ\*, y un Poisson adelgazado así es un Poisson de tasa λ(t), sea cual sea
  la cota mientras domine a λ.
- **Los huecos en minutos no son exponenciales** porque mezclan tasas distintas a lo largo del
  día; solo reescalados con Λ son Exp(1).
- **Se calibra el CV y no el índice de dispersión** porque el índice, 1 + CV²·ē, crece con el
  volumen del centro, mientras que el CV se puede trasladar de un centro a otro.